In [ ]:
import joblib

PRODUCTS_DATA = joblib.load('data/clothes_json.joblib')

In [3]:
PRODUCTS_DATA[0].keys()

dict_keys(['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'price', 'product_id'])

In [4]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [ ]:
FAQ = joblib.load("data/faq.joblib")

In [6]:
print(FAQ[0].keys())
FAQ[0]

dict_keys(['question', 'answer', 'type'])


{'question': 'What are your store hours?',
 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
 'type': 'general information'}

In [ ]:
import ollama
import json

def check_if_faq_or_product(query: str) -> str:
    system_prompt = (
        'You are a strict text classification API. Classify the user\'s query into exactly one of these three classes: "FAQ", "Product", or "OTHER".\n\n'
        'You must output ONLY a single word representing the class and nothing else.\n\n'
        '- Use "FAQ" for policies, instructions, support, or general system questions.\n'
        '- Use "Product" for item availability, prices, recommendations, comparisons, or styling/outfit requests.\n'
        '- Use "OTHER" for greetings, general conversation, chit-chat, or topics unrelated to fashion products and company policies.'
    )

    messages = [
        {'role': 'system', 'content': system_prompt},
        # {'role': 'user', 'content': 'How do I reset my password?'},
        # {'role': 'assistant', 'content': '{"category": "FAQ"}'},
        # {'role': 'user', 'content': 'Do you have blue T-shirts under 100 dollars?'},
        # {'role': 'assistant', 'content': '{"category": "Product"}'},
        # {'role': 'user', 'content': 'what is the result of 5x + 5 = 25 '},
        # {'role': 'assistant', 'content': '{"category": "OTHER"}'},
        # {'role': 'user', 'content': 'how is the weather today'},
        # {'role': 'assistant', 'content': '{"category": "OTHER"}'},
        # {'role': 'user', 'content': 'Create a look suitable for a wedding party.'},
        # {'role': 'assistant', 'content': '{"category": "Product"}'},
        # {'role': 'user', 'content': 'I have a job interview tomorrow, what do you recommend I wear?'},
        # {'role': 'assistant', 'content': '{"category": "Product"}'},
        {'role': 'user', 'content': query},
    ]

    response = ollama.chat(
        model='deepseek-r1:7b',
        messages=messages,
        # think=False,       
        options={
            'temperature': 0.1,
            'num_predict': 700,   
            
        }
    )
    return response.message.content.strip()

In [156]:
queries = [
      'What is your return policy?', 
         'Give me three examples of blue T-shirts you have available.', 
         'How can I contact the user support?', 
         'Do you have blue Dresses?',
         'Create a look suitable for a wedding party happening during dawn.',
         'how are you ?',
         'Who won the basketball game yesterday',
         'I lost my password, how can I access my account',
         'Do you have anything to keep me warm in the winter?',
         'what is the result of 5x + 5 = 25',
         'I have a job interview tomorrow, what do you recommend I wear?'
]

for query in queries:
   response = check_if_faq_or_product(query)
   print(f"Query: {query} Label: {response}")

Query: What is your return policy? Label: FAQ
Query: Give me three examples of blue T-shirts you have available. Label: Product
Query: How can I contact the user support? Label: FAQ
Query: Do you have blue Dresses? Label: Product
Query: Create a look suitable for a wedding party happening during dawn. Label: OTHER
Query: how are you ? Label: OTHER
Query: Who won the basketball game yesterday Label: OTHER
Query: I lost my password, how can I access my account Label: FAQ
Query: Do you have anything to keep me warm in the winter? Label: Product
Query: what is the result of 5x + 5 = 25 Label: OTHER
Query: I have a job interview tomorrow, what do you recommend I wear? Label: Product


In [9]:
len(FAQ)

25

In [10]:
def generate_faq_layout(faq_dict: list) -> str:
    """
    Generates a formatted string layout for a list of FAQs.

    This function iterates through a dictionary of frequently asked questions (FAQs) and constructs
    a string where each question is followed by its corresponding answer and type.

    Parameters:
    - faq_dict (list): A list of dictionaries, each containing keys 'question', 'answer', and 'type' 
      representing an FAQ entry.

    Returns:
    - str: A string representing the formatted layout of FAQs, with each entry on a separate line.
    """
    text = " "
    for f in faq_dict:
        text += f'question: {f["question"]} Answer: {f["answer"]} Type: {f["type"]}\n '

    return text 

In [11]:
faq_layout = generate_faq_layout(FAQ)

In [12]:
print(len(faq_layout.split(" ")))
print(faq_layout[:1000])

724
 question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information
 question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information
 question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information
 question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information
 question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information


In [159]:
import ollama


def query_on_faq(query: str) -> str:

    system_prompt = """
    You are a professional FAQ question-answering assistant.

    Your job is to answer the user's question using the FAQ context.

    """

    user_prompt = f"""
    Here is the FAQ context:

    {faq_layout}

    Here is the user's question:

    {query}


    Answer the user's question using only the FAQ context.
    """
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        options={
            "temperature": 0.,
            "num_predict": 200
        }
    )

    return response.message.content.strip()

In [ ]:
faq_layout = generate_faq_layout(FAQ)
res = query_on_faq("I got my cloth but I didn't like it. How can I return it")

In [162]:
print(res)

To initiate a return, please follow these steps:

1. Go to our Returns Center and select the item you wish to exchange or return.
2. Choose the reason for return (e.g., "Item didn't fit" or "Item wasn't as described").
3. Select the desired replacement or request a refund.

Return processing typically takes 5-7 business days from when the item is received at our warehouse.

Please note that sale items are final sale and cannot be returned or exchanged, unless stated otherwise.


In [20]:
faq_layout

" question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information\n question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information\n question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information\n question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information\n question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information

In [ ]:
def decide_task_nature(query: str) -> str:
    system_prompt = """Decide if the following query is a query that requires creativity (creating, composing, making new things) or technical (information about products, prices, etc.). 
    Label it as creative or technical.

    Examples:
    Query: Give me suggestions on a nice look for a nightclub.
    Label: creative

    Query: What are the blue dresses you have available?
    Label: technical

    Query: Give me three T-shirts for summer.
    Label: technical

    Query: Give me a look for attending a wedding party.
    Label: creative

    Only output one token: the label."""

    user_prompt = f"""
    question:
    
    {query}

    """
    messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
            
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        options={
            "temperature": 0.1,
            "num_predict": 200
        }
    )

    return response.message.content.strip()

In [22]:
queries = ["Give me two sneakers with vibrant colors.",
           "What are the most expensive clothes you have in your catalogue?",
           "I have a green dress and I like a suggestion on an accessory to match with it.",
           "Give me three trousers with vibrant colors you have in your catalogue.",
           "Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather."
           ]
for query in queries:
    label = decide_task_nature(query)
    print(f"Query: {query} Label: {label}")

Query: Give me two sneakers with vibrant colors. Label: technical
Query: What are the most expensive clothes you have in your catalogue? Label: technical
Query: I have a green dress and I like a suggestion on an accessory to match with it. Label: creative
Query: Give me three trousers with vibrant colors you have in your catalogue. Label: technical
Query: Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather. Label: creative


In [23]:
def get_params_for_task(task: str) -> dict:
    
    PARAMETERS_DICT = {
        "creative": {"top_p": 0.7, 'temperature': 1.2},
        "technical": {'top_p': 0.9, 'temperature': 0.1}
    }

    return PARAMETERS_DICT.get(task, PARAMETERS_DICT["technical"])

In [24]:
get_params_for_task("technical")

{'top_p': 0.9, 'temperature': 0.1}

In [25]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [26]:
values = {}
for d in PRODUCTS_DATA:
    for key, val in d.items():
        if key in ('product_id', 'price', 'productDisplayName', 'subCategory', 'year'):
            continue
        if key not in values.keys():
            values[key] = set()
        values[key].add(val)

In [27]:
values['season']

{'All seasons', 'Fall', 'Spring', 'Summer', 'Winter'}

In [28]:
values['gender']

{'Boys', 'Girls', 'Men', 'Unisex', 'Women'}

In [ ]:
def generate_metadata_from_query(query: str) -> str:
    system_prompt = f"""Given user query, your task is extract the following information and output in JSON format by strictly following given instructions.
    Information to extract:
    - **Gender**: Target audience for the product, such as "Men," "Women," or "Unisex."
    - **Master Category**: Broad classification like "Apparel" or "Footwear."
    - **Article Type**: Exact type of product, e.g., "Shirts" or "Jackets."
    - **Base Colour**: Main color of the product, important for customer choice.
    - **Season**: Intended season for the product, e.g., "Summer" or "Winter."
    - **Usage**: Intended use or occasion, like "Casual" or "Formal."
    - **Price**: Cost of the product.
    Instructions: 
    - Extract all information explained above
    - strictly use these keys names for JSON output: "gender", "masterCategory", "articleType", "baseColour", "price", "usage", "season".
    - values must be of list of string except 'price'. 'price' value must dictionary with two keys "min" and "max". If no 'price' mentioned, set "min" to 0 and "max" to "inf".
    - Only return Valid JSON output without anything else. 
    - here is example output json format.
    {{
        "gender": ["Women"],
        "masterCategory": ["Apparel"],
        "articleType": ["Dresses"],
        "baseColour": ["Blue"],
        "price": {{"min": 0, "max": "inf"}},
        "usage": ["Formal"],
        "season": ["All seasons"]
    }}
    
    """
    user_prompt = f"""
    query: 
    {query}
    """
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
            
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        format= "json",
        options={
            "temperature": 0.1,
            "num_predict": 200
        }
    )
    return response.message.content.strip()

In [ ]:
def parse_json_output(llm_output: str) -> dict:
    
    try:
        llm_output = llm_output.replace("\n", '').replace("'",'').replace("}}", "}").replace("{{", "{")  # Remove any erroneous structures
        
        parsed_json = json.loads(llm_output)
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return None

In [31]:
output = generate_metadata_from_query("Create a look for a man that suits a sunny day in the park. I don't want to spend more than 300 dollars on each piece." )

In [32]:
parse_json_output(output)

{'gender': ['Men'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts', 'Shorts'],
 'baseColour': ['Navy Blue', 'Light Grey'],
 'price': {'min': 0, 'max': 300},
 'usage': ['Casual'],
 'season': ['Summer']}

In [33]:
json_string = generate_metadata_from_query("I need men shirt of 2xl, with price range 100 to 1000, for summer season in black color")
json_output = parse_json_output(json_string)
json_output

{'gender': ['Men'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts'],
 'baseColour': ['Black'],
 'price': {'min': 100, 'max': 1000},
 'usage': ['Casual'],
 'season': ['Summer']}

In [34]:
json_string = generate_metadata_from_query("I need shirt for baby")
json_output = parse_json_output(json_string)
json_output

{'gender': ['Baby'],
 'masterCategory': ['Apparel'],
 'articleType': ['Shirts'],
 'baseColour': ['White', 'Pastel colors'],
 'price': {'min': 0, 'max': 'inf'},
 'usage': ['Casual'],
 'season': ['All seasons']}

In [35]:
print(len(PRODUCTS_DATA))

44424


In [39]:
PRODUCTS_DATA[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011.0,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67,
 'product_id': 15970}

In [ ]:
def clean_record(record):

    cleaned = {}
    
    text_fields = ["gender", "masterCategory", "subCategory", "articleType", 
                   "usage", "season", "productDisplayName", "baseColour"]
    
    for field in text_fields:
        val = record[field]
        if val is None or str(val).strip() == "" or str(val).lower() == "nan":
            cleaned[field] = None
        else:
            cleaned[field] = str(val).strip()

    try:
        price_val = record.get("price")
        if price_val is None or str(price_val).strip() == "":
            cleaned["price"] = None
        else:
            cleaned["price"] = float(price_val)
    except ValueError:
        cleaned["price"] = None 

    try:
        id_val = record["product_id"]
        if id_val is None or str(id_val).strip() == "":
            cleaned["product_id"] = None
        else:
            cleaned["product_id"] = int(id_val)
    except ValueError:
        cleaned["product_id"] = None

    return cleaned

In [ ]:
import weaviate

client = weaviate.connect_to_local(port=8090, grpc_port=50051)

client.is_ready()

True

In [48]:
from weaviate.classes.config import Configure, Property, DataType

if not client.collections.exists("products"):
    collections = client.collections.create(
        name= "products",

        vectorizer_config=Configure.Vectorizer.text2vec_ollama(
            model="nomic-embed-text",
            api_endpoint="http://host.docker.internal:11434",
            vectorize_collection_name=False
             
        ),
            properties = [
            Property(name="productDisplayName", data_type=DataType.TEXT),
            Property(name="articleType", data_type=DataType.TEXT),
            Property(name="baseColour", data_type=DataType.TEXT),
            Property(name="usage", data_type=DataType.TEXT),
            Property(name="season", data_type=DataType.TEXT),
            
            Property(name="price", data_type=DataType.NUMBER, skip_vectorization=True),
            Property(name="product_id", data_type=DataType.INT, skip_vectorization=True),
            Property(name="year", data_type=DataType.TEXT, skip_vectorization=True),
            Property(name="gender", data_type=DataType.TEXT, skip_vectorization=True) 
        ]
    )
else : 
    collections = client.collections.get("products")

c:\Users\khale\anaconda3\envs\RAG\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [ ]:
import tqdm
collection = client.collections.get("products")

with collection.batch.dynamic() as batch:
    for record in tqdm.tqdm(PRODUCTS_DATA):
        cleaned_properties = clean_record(record)
        
        batch.add_object(
            properties=cleaned_properties
        )
        
if len(collection.batch.failed_objects) > 0:
    print(f"{len(collection.batch.failed_objects)}")
    print(collection.batch.failed_objects[0].message)
else:
    print("done ")

  0%|          | 0/44424 [00:00<?, ?it/s]

100%|██████████| 44424/44424 [07:07<00:00, 104.01it/s]


done 


In [164]:
print(len(collection))

44616


In [ ]:
from weaviate.classes.query import Filter
def get_filter_by_metadata(json_output: dict | None = None):

    if json_output == None:
        return None
    
    valid_keys = (
        'gender',
        'masterCategory',
        'articleType',
        'baseColour',
        'price',
        'usage',
        'season',
    )

    filters = []
    for key, value in json_output.items():

        if key not in valid_keys:
            continue

        if key == "price":

            if not isinstance(value, dict):
                continue

            max_price = value.get('max')
            min_price = value.get('min')

            if max_price == None or min_price == None:
                continue

            if min_price > 0:
                filters.append(Filter.by_property(key).greater_or_equal(min_price))
            
            if max_price != 'inf':
                filters.append(Filter.by_property(key).less_or_equal(max_price))

        else:
            filters.append(Filter.by_property(key).contains_any(value))

    return filters

In [53]:
def generate_filters_from_query(query: str) -> list:
    json_string = generate_metadata_from_query(query)
    json_output = parse_json_output(json_string)
    filters = get_filter_by_metadata(json_output)
    return filters

In [ ]:
filters = generate_filters_from_query("Give me three T-shirts to use in sunny days")
filters 


[_FilterValue(value=['Men', 'Women'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='gender'),
 _FilterValue(value=['Apparel'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='masterCategory'),
 _FilterValue(value=['T-shirts'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='articleType'),
 _FilterValue(value=['White', 'Light Blue', 'Yellow'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='baseColour'),
 _FilterValue(value=['Casual'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='usage'),
 _FilterValue(value=['Summer'], operator=<_Operator.CONTAINS_ANY: 'ContainsAny'>, target='season')]

In [59]:
filters[-1].target

'season'

In [61]:
from weaviate.classes.query import Filter

def get_relevant_products_from_query(query: str):
    filters = generate_filters_from_query(query)

    if filters is None or len(filters) == 0:
        response = collections.query.near_text(
            query=query,
            limit=20,
        ).objects 
        return response
    
    response = collections.query.near_text(
        query=query,
        limit=20,
        filters=Filter.all_of(filters)
    ).objects

    if len(response) >= 10:
        return response

    importance_order = [
        'gender',          
        'masterCategory',  
        'articleType',     
        'price',           
        'baseColour',     
        'season',        
        'usage',      
        'year'        
    ]
    
    for i in range(1, len(importance_order)):
        
        keys_to_keep = importance_order[:-i] 

        current_filters = [f for f in filters if f.target in keys_to_keep]

        if len(current_filters) == 0:
            break

        response = collections.query.near_text(
            query=query,
            limit=20,
            filters=Filter.all_of(current_filters) 
        ).objects
        
        if len(response) >= 5:
            return response

    response = collections.query.near_text(
        query=query,
        limit=20,
    ).objects
    
    return response

In [67]:
test = get_relevant_products_from_query("Give me three T-shirts to use in sunny days")

In [71]:
for i in test:
    print(i.properties)

{'gender': 'Men', 'price': 107.0, 'usage': 'Casual', 'articleType': 'Tshirts', 'baseColour': 'White', 'season': 'Summer', 'product_id': 29788, 'year': '2013.0', 'subCategory': 'Topwear', 'masterCategory': 'Apparel', 'productDisplayName': 'Basics Men Pack of 3 T-shirts'}
{'gender': 'Women', 'price': 72.0, 'masterCategory': 'Apparel', 'articleType': 'Tshirts', 'baseColour': 'Peach', 'season': 'Summer', 'product_id': 47285, 'year': '2012.0', 'usage': 'Casual', 'subCategory': 'Topwear', 'productDisplayName': 'Myntra Women Pack of 3 T-shirts'}
{'gender': 'Men', 'price': 250.0, 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'baseColour': 'Beige', 'season': 'Summer', 'product_id': 37152, 'year': '2016.0', 'usage': 'Casual', 'articleType': 'Tshirts', 'productDisplayName': 'Campbell Men Pack of 3 T-shirts'}
{'gender': 'Men', 'price': 253.0, 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'baseColour': 'Pink', 'season': 'Summer', 'product_id': 12225, 'year': '2013.0', 'articleType

In [75]:
def generate_items_context(results: list) -> str:
    res = ""
    for obj in results:
        properties = obj.properties
        res += (
            f"Product ID: {properties.get('product_id')}, "
            f"Product: {properties.get('productDisplayName')}, "
            f"Price: {properties.get('price')}, "
            f"Color: {properties.get('baseColour')}, "
            f"Category: {properties.get('articleType')}, "
            f"Gender: {properties.get('gender')}, "
            f"Season: {properties.get('season')}.\n"
            
        )
    return res

In [76]:
print(generate_items_context(test)[:1000])

Product ID: 29788, Product: Basics Men Pack of 3 T-shirts, Price: 107.0, Color: White, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 47285, Product: Myntra Women Pack of 3 T-shirts, Price: 72.0, Color: Peach, Category: Tshirts, Gender: Women, Season: Summer.
Product ID: 37152, Product: Campbell Men Pack of 3 T-shirts, Price: 250.0, Color: Beige, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 12225, Product: Basics Men Pack of 3 T-shirts, Price: 253.0, Color: Pink, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 47282, Product: Myntra Men Pack of 3 T-shirts, Price: 125.0, Color: Pink, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 20284, Product: Campbell Men Pack of 3 T-shirts, Price: 150.0, Color: Green, Category: Tshirts, Gender: Men, Season: Summer.
Product ID: 7891, Product: Proline Men Pack of 3 T-shirts, Price: 172.0, Color: Cream, Category: Tshirts, Gender: Men, Season: Fall.
Product ID: 24818, Product: Basics Men Pack of 3 T-s

In [77]:
import ollama

def query_on_products(query: str) -> str: 
    query_label = decide_task_nature(query)
    parameters_dict = get_params_for_task(query_label)
    relevant_products = get_relevant_products_from_query(query)
    context = generate_items_context(relevant_products)

    system_prompt = (
        "Given the available set of cloth products, answer the question that follows, providing the item ID in your answers. "
        "Other information might be provided but not necessarily all of them; pick only the relevant ones for the given query and avoid being too long when describing the items' features. "
        "If no number of products is mentioned in the query, select at most five to show. "
        "Act as a helpful fashion assistant."
    )

    user_prompt = (
        f"CLOTH PRODUCTS AVAILABLE: \n{context}\n\n"
        f"QUERY: {query}"
    )
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        options={
            "temperature": parameters_dict.get('temperature', 0.7),
            "top_p": parameters_dict.get('top_p', 0.9),
            "num_predict": 300 
        }
    )
    
    return response['message']['content'].strip()

In [157]:
import ollama

chat_memory = []

def answer_query(query: str):
    global chat_memory 
    
    label = check_if_faq_or_product(query)
    final_response = "" 

    if label == "FAQ":
        final_response = query_on_faq(query)

    elif label == "Product":
        try:
            final_response = query_on_products(query)
        except Exception as e:
            system_prompt = (
                "User provided a question that broke the querying system. Instruct them to rephrase it. "
                "Answer it based on the context you already have so far."
            )
            user_prompt = f"User question {query}"

            messages = [{'role': 'system', 'content': system_prompt}]
            messages.extend(chat_memory) 
            messages.append({'role': 'user', 'content': user_prompt})

            response = ollama.chat(
                model="llama3.2",
                messages=messages,
                options={
                    "temperature": 0.7,
                    "num_predict": 300 
                }
            )
            final_response = response['message']['content'].strip() 

    else:
        system_prompt = (
            "You are a helpful assistant. The user provided a question that does not fit FAQ or Product related questions. "
            "Answer it based on the context you already have so far."
        )
        user_prompt = f"user question {query}"

        messages = [{'role': 'system', 'content': system_prompt}]
        messages.extend(chat_memory)
        messages.append({'role': 'user', 'content': user_prompt})
            
        response = ollama.chat(
            model="llama3.2",
            messages=messages,
            options={
                "temperature": 0.7,
                "num_predict": 300 
            }
        )
        final_response = response['message']['content'].strip() 
    
    chat_memory.append({'role': 'user', 'content': query})
    chat_memory.append({'role': 'assistant', 'content': final_response})
    
    if len(chat_memory) > 6:
        chat_memory = chat_memory[-6:]
        
    return final_response

In [160]:
test1 = answer_query("What are your working hours?")
test2 = answer_query("Tomorrow is my engagement day Suggest what to wear")

In [ ]:
print(test1)
print("-" * 50)
# print(test2)

I'm an artificial intelligence language model, and I don't have traditional working hours like humans do. My system is designed to be available 24/7, meaning I can respond to queries at any time.

However, my responses may vary in terms of quality or accuracy depending on the complexity of the question, the amount of data required to provide a relevant answer, and other factors.

That being said, I don't require rest or breaks like humans do. My training data is constantly updated and refreshed, which means that I can access and process vast amounts of information at any given time.

So, feel free to ask me anything, day or night!
--------------------------------------------------
What a wonderful occasion!

Considering it's an engagement day, I'd recommend something elegant and special. Based on the available products, here are five suggestions for you:

1. **French Connection Women White Dress** (Product ID: 43678) - A classic white dress that exudes simplicity and sophistication.
2.